In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.stats.multitest as smm
from statsmodels.stats.multitest import multipletests

In [2]:
ldsc = pd.read_csv("/home/maria/git/SOROLLA/Results/ldsc_genetic_correlation.csv")
hdl = pd.read_csv("/home/maria/git/SOROLLA/Results/hdl_genetic_correlation.csv")
excell = pd.read_csv("/home/maria/git/SOROLLA/SumStats/excell_sumstats_description.csv")

In [3]:
display(ldsc)

,id_1,label_1,id_2,label_2,rg,se,z,p,gcov_int,gcov_int_se,h2_1,h2_1_se,h2_2,h2_2_se
0,GCST90275158,schizophrenia-M1,GCST90013410,bcc-1,-0.0130,0.0517,-0.2519,0.8011,0.000400,0.0057,0.1280,0.0131,0.0317,0.0044
1,GCST90042698,lung-cancer-family-sibling,GCST90041853,prostate-cancer-5,0.0827,0.1500,0.5509,0.5817,-0.002800,0.0045,0.0043,0.0014,0.0180,0.0029
2,GCST90011816,cervical-cancer-2,GCST90013410,bcc-1,0.0549,0.0829,0.6613,0.5084,0.036300,0.0055,0.0089,0.0025,0.0315,0.0044
3,GCST90042843,anxiety-treatment-1,GCST90271619,partial-epilepsy-2,-0.1741,0.1330,-1.3089,0.1906,0.008200,0.0071,0.0383,0.0049,0.0468,0.0111
4,GCST003740,adenocarcinoma-barret-oesophagus,GCST90137411,bcc-2,0.0679,0.0479,1.4178,0.1563,0.003000,0.0061,0.2387,0.0309,0.0697,0.0121
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6550,GCST90041833,breast-neoplasm-2,GCST90271619,partial-epilepsy-2,-0.2053,0.1419,-1.4469,0.1479,0.005600,0.0072,0.0082,0.0018,0.0469,0.0112
6551,GCST009979,mdd-1,GCST90435597,skin-neoplasm,-0.0353,0.0585,-0.6038,0.5460,0.000029,0.0055,0.0790,0.0060,0.0171,0.0030
6552,GCST007511,alzheimers-late-onset-1,GCST90275149,ADHD-1,0.0530,0.0617,0.8597,0.3900,0.002000,0.0056,0.0540,0.0136,0.2441,0.0144
6553,GCST008465,ANN-PGC-2019,GCST90137411,bcc-2,0.0543,0.0366,1.4844,0.1377,-0.003300,0.0081,0.1606,0.0134,0.0680,0.0149


In [4]:
ldsc.columns

Index(['id_1', 'label_1', 'id_2', 'label_2', 'rg', 'se', 'z', 'p', 'gcov_int',
       'gcov_int_se', 'h2_1', 'h2_1_se', 'h2_2', 'h2_2_se'],
      dtype='object')

In [5]:
excell.columns

Index(['type', 'selection', 'title', 'pubmedId', 'efoTraits', 'disease',
       'disease_subtype', 'label', 'condition_label', 'summaryStatistics',
       'id', 'filename', 'Ancestry', 'ref_genome', 'snp', 'a1', 'a2', 'frq',
       'FRQ_U', 'FRQ_A', 'z', 'b', 'OR', 'se', 'p', 'N_col', 'N_num',
       'Nca_col', 'Nca_val', 'Nco_col', 'Nco_val', 'INFO', 'ignore'],
      dtype='object')

In [6]:
# Define the wanted columns for merging 
dfcols = excell[["type","selection","disease","disease_subtype","label","id","pubmedId", "N_num", "Nca_val", "Nco_val"]]

# Merge to get type_1
ldsc = pd.merge(ldsc, dfcols, how='left', left_on=['label_1','id_1'], right_on=['label','id'])
ldsc.rename(columns={'type': 'type_1', 'selection':'selection_1','disease':'disease_1','disease_subtype':'disease_subtype_1','pubmedId':'pubmedId_1', 'N_num':'N_num_1', 'Nca_val':'Nca_val_1', 'Nco_val':'Nco_val_1'}, inplace=True)
ldsc.drop(columns=['label'], inplace=True) 
ldsc.drop(columns=['id'], inplace=True) 

# Merge to get type_2
ldsc = pd.merge(ldsc, dfcols, how='left', left_on=['label_2','id_2'], right_on=['label','id'])
ldsc.rename(columns={'type': 'type_2', 'selection':'selection_2','disease':'disease_2','disease_subtype':'disease_subtype_2','pubmedId':'pubmedId_2', 'N_num':'N_num_2', 'Nca_val':'Nca_val_2', 'Nco_val':'Nco_val_2'}, inplace=True)
ldsc.drop(columns=['label'], inplace=True) 
ldsc.drop(columns=['id'], inplace=True) 

In [7]:
ldsc = ldsc[['id_1', 'label_1','disease_1','type_1','selection_1', 'disease_subtype_1','pubmedId_1',
			 'id_2', 'label_2', 'disease_2','type_2','selection_2',  'disease_subtype_2', 'pubmedId_2',
			 'rg', 'se', 'z', 'p', 
			 'N_num_1', 'Nca_val_1', 'Nco_val_1',
			 'N_num_2', 'Nca_val_2', 'Nco_val_2',
			 'gcov_int',
			 'gcov_int_se', 'h2_1', 'h2_1_se', 'h2_2', 'h2_2_se']]

In [8]:
ldsc.columns

Index(['id_1', 'label_1', 'disease_1', 'type_1', 'selection_1',
       'disease_subtype_1', 'pubmedId_1', 'id_2', 'label_2', 'disease_2',
       'type_2', 'selection_2', 'disease_subtype_2', 'pubmedId_2', 'rg', 'se',
       'z', 'p', 'N_num_1', 'Nca_val_1', 'Nco_val_1', 'N_num_2', 'Nca_val_2',
       'Nco_val_2', 'gcov_int', 'gcov_int_se', 'h2_1', 'h2_1_se', 'h2_2',
       'h2_2_se'],
      dtype='object')

In [9]:
display(hdl)

,SNP_rm_GWAS1_Nmiss,SNP_rm_GWAS2_Nmiss,SNP_RP_GWAS1_TOTAL,SNP_RP_GWAS1_PERC,SNP_RP_GWAS2_TOTAL,SNP_RP_GWAS2_PERC,h2_1,h2_1_se,h2_2,h2_2_se,...,gcov_se,rg,se,p,output_file,output_file_path,id_1,label_1,id_2,label_2
0,0,0,972817,1029876,907187,1029876,0.0320,0.0079,0.0592,0.0050,...,0.0030,0.4668,0.0852,4.240000e-08,GCST004744_adenocarcinoma-lung_GCST90137412_sq...,/gpfs/projects/bsc02/mflores/gencor/Results/HD...,GCST004744,adenocarcinoma-lung,GCST90137412,squamous-carcinoma
1,0,0,1029867,1029876,971756,1029876,0.0043,0.0009,0.0227,0.0083,...,0.0019,0.4506,0.2240,4.420000e-02,GCST90042685_intestinal-cancer-mother_GCST0047...,/gpfs/projects/bsc02/mflores/gencor/Results/HD...,GCST90042685,intestinal-cancer-mother,GCST004750,lung-cancer-squamous
2,0,336,1008231,1029876,836400,1029876,0.0366,0.0143,0.1216,0.0106,...,0.0051,0.2500,0.1024,1.470000e-02,GCST002245_alzheimers-late-onset-4_GCST9027161...,/gpfs/projects/bsc02/mflores/gencor/Results/HD...,GCST002245,alzheimers-late-onset-4,GCST90271613,partial-epilepsy-1
3,0,0,1029738,1029876,972817,1029876,0.0058,0.0010,0.0320,0.0079,...,0.0019,0.2304,0.1454,1.130000e-01,GCST90435919_headache-disorder-1_GCST004744_ad...,/gpfs/projects/bsc02/mflores/gencor/Results/HD...,GCST90435919,headache-disorder-1,GCST004744,adenocarcinoma-lung
4,0,0,1029721,1029876,1016163,1029876,0.3665,0.0092,0.0201,0.0021,...,0.0022,0.0242,0.0252,3.370000e-01,GCST90128471-all_SCZ-PGC-2022-all_GCST90239856...,/gpfs/projects/bsc02/mflores/gencor/Results/HD...,GCST90128471-all,SCZ-PGC-2022-all,GCST90239856,uterine-leiomyoma-2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6550,0,0,1029867,1029876,1029335,1029876,0.0045,0.0012,0.0000,0.0000,...,0.0010,inf,NaN,NaN,GCST90042700_breast-cancer-family-siblings_GCS...,/gpfs/projects/bsc02/mflores/gencor/Results/HD...,GCST90042700,breast-cancer-family-siblings,GCST90428116,breast-cancer-4
6551,0,0,1027244,1029876,1029867,1029876,0.0115,0.0008,0.0073,0.0010,...,0.0006,0.7454,0.0881,2.740000e-17,GCST90225529_anxiety-others_GCST90042669_depre...,/gpfs/projects/bsc02/mflores/gencor/Results/HD...,GCST90225529,anxiety-others,GCST90042669,depression-family-father
6552,0,0,1029867,1029876,1027982,1029876,0.0138,0.0011,0.0760,0.0052,...,0.0018,0.8397,0.0764,4.140000e-28,GCST90042695_depression-family-sibling_GCST010...,/gpfs/projects/bsc02/mflores/gencor/Results/HD...,GCST90042695,depression-family-sibling,GCST010009,mdd-lifetime-1
6553,0,0,1029335,1029876,1029738,1029876,0.0000,0.0000,0.0058,0.0010,...,0.0012,-inf,NaN,NaN,GCST90428116_breast-cancer-4_GCST90435919_head...,/gpfs/projects/bsc02/mflores/gencor/Results/HD...,GCST90428116,breast-cancer-4,GCST90435919,headache-disorder-1


In [10]:
hdl.columns

Index(['SNP_rm_GWAS1_Nmiss', 'SNP_rm_GWAS2_Nmiss', 'SNP_RP_GWAS1_TOTAL',
       'SNP_RP_GWAS1_PERC', 'SNP_RP_GWAS2_TOTAL', 'SNP_RP_GWAS2_PERC', 'h2_1',
       'h2_1_se', 'h2_2', 'h2_2_se', 'gcov', 'gcov_se', 'rg', 'se', 'p',
       'output_file', 'output_file_path', 'id_1', 'label_1', 'id_2',
       'label_2'],
      dtype='object')

In [11]:
# Merge to get type_1
hdl = pd.merge(hdl, dfcols, how='left', left_on=['label_1','id_1'], right_on=['label','id'])
hdl.rename(columns={'type': 'type_1', 'selection':'selection_1','disease':'disease_1','disease_subtype':'disease_subtype_1','pubmedId':'pubmedId_1', 'N_num':'N_num_1', 'Nca_val':'Nca_val_1', 'Nco_val':'Nco_val_1'}, inplace=True)
hdl.drop(columns=['label'], inplace=True) 
hdl.drop(columns=['id'], inplace=True) 

# Merge to get type_2
hdl = pd.merge(hdl, dfcols, how='left', left_on=['label_2','id_2'], right_on=['label','id'])
hdl.rename(columns={'type': 'type_2', 'selection':'selection_2','disease':'disease_2','disease_subtype':'disease_subtype_2','pubmedId':'pubmedId_2', 'N_num':'N_num_2', 'Nca_val':'Nca_val_2', 'Nco_val':'Nco_val_2'}, inplace=True)
hdl.drop(columns=['label'], inplace=True) 
hdl.drop(columns=['id'], inplace=True) 

In [12]:
hdl.columns

Index(['SNP_rm_GWAS1_Nmiss', 'SNP_rm_GWAS2_Nmiss', 'SNP_RP_GWAS1_TOTAL',
       'SNP_RP_GWAS1_PERC', 'SNP_RP_GWAS2_TOTAL', 'SNP_RP_GWAS2_PERC', 'h2_1',
       'h2_1_se', 'h2_2', 'h2_2_se', 'gcov', 'gcov_se', 'rg', 'se', 'p',
       'output_file', 'output_file_path', 'id_1', 'label_1', 'id_2', 'label_2',
       'type_1', 'selection_1', 'disease_1', 'disease_subtype_1', 'pubmedId_1',
       'N_num_1', 'Nca_val_1', 'Nco_val_1', 'type_2', 'selection_2',
       'disease_2', 'disease_subtype_2', 'pubmedId_2', 'N_num_2', 'Nca_val_2',
       'Nco_val_2'],
      dtype='object')

In [13]:
hdl= hdl[['id_1', 'label_1','type_1', 'selection_1', 'disease_1', 'disease_subtype_1', 'pubmedId_1',
		  'id_2', 'label_2','type_2', 'selection_2', 'disease_2', 'disease_subtype_2', 'pubmedId_2',
		  'h2_1', 'h2_1_se', 'h2_2', 'h2_2_se', 'N_num_1', 'Nca_val_1', 'Nco_val_1',
		  'N_num_2', 'Nca_val_2', 'Nco_val_2',
		  'gcov', 'gcov_se', 'rg', 'se', 'p', 
		  'output_file', 'output_file_path',
		  'SNP_rm_GWAS1_Nmiss', 'SNP_rm_GWAS2_Nmiss', 'SNP_RP_GWAS1_TOTAL',
		  'SNP_RP_GWAS1_PERC', 'SNP_RP_GWAS2_TOTAL', 'SNP_RP_GWAS2_PERC', ]]

#### LDSC correction

In [14]:
#Bonferroni correction
reject_bonf_ldsc, corrected_p_vals_ldsc, _, _ = smm.multipletests(ldsc['p'], method='bonferroni')

#Replace original 'p' column with corrected p-values
ldsc['p_corrected_bonf'] = corrected_p_vals_ldsc
ldsc['p_bonf_rejected'] = reject_bonf_ldsc

In [15]:
# False Discovery Rate (FDR)
reject_FDR_ldsc, corrected_FDR_ldsc = smm.fdrcorrection(ldsc["p"], method='indep', is_sorted=False)

#Replace original 'p' column with corrected p-values
ldsc['p_corrected_FDR'] = corrected_FDR_ldsc
ldsc['p_FDR_rejected'] = reject_FDR_ldsc

In [16]:
print(f"The number of disease-pairs that are relevant without correction are", {len(ldsc[ldsc["p"]<0.05])},"out of", {len(ldsc)})
print(f"And the number of pairs that were bonferroni accepted are",len(ldsc[ldsc["p_bonf_rejected"] == True]))
print(f"And the number of pairs that were FDR accepted are",len(ldsc[ldsc["p_FDR_rejected"] == True]))

The number of disease-pairs that are relevant without correction are {2329} out of {6555}
And the number of pairs that were bonferroni accepted are 1089
And the number of pairs that were FDR accepted are 1912


In [17]:
ldsc.to_csv('/home/maria/git/SOROLLA/Results/ldsc_genetic_correlation_pcorrected.csv', index=False)


#### HDL correction

In [18]:
#Bonferroni correction
reject_bonf_hdl, corrected_p_vals_hdl, _, _ = smm.multipletests(hdl['p'], method='bonferroni')

#Replace original 'p' column with corrected p-values
hdl['p_corrected_bonf'] = corrected_p_vals_hdl
hdl['p_bonf_rejected'] = reject_bonf_hdl

In [19]:
# FDR does not work with NAs, so we will fill p_value with 1
hdl["missing_p"] = hdl["p"].isna()
hdl[["p"]] = hdl[["p"]].fillna(value=1)

In [20]:
# False Discovery Rate (FDR)
reject_FDR_hdl, corrected_FDR_hdl = smm.fdrcorrection(hdl["p"], method='indep', is_sorted=False)

#Replace original 'p' column with corrected p-values
hdl['p_corrected_FDR'] = corrected_FDR_hdl
hdl['p_FDR_rejected'] = reject_FDR_hdl

In [21]:
p_value_threshold_hdl = 0.05/len(hdl)
print(f"The p-value threshold after bonferroni correction is:",p_value_threshold_hdl)

The p-value threshold after bonferroni correction is: 7.627765064836003e-06


In [22]:
print(f"The number of disease-pairs that are relevant without correction are", {len(hdl[hdl["p"]<0.05])},"out of", {len(hdl)})
print(f"And the number of pairs that were bonferroni accepted are",len(hdl[hdl["p_bonf_rejected"] == True]))
print(f"And the number of pairs that were FDR accepted are",len(hdl[hdl["p_FDR_rejected"] == True]))

The number of disease-pairs that are relevant without correction are {2747} out of {6555}
And the number of pairs that were bonferroni accepted are 1412
And the number of pairs that were FDR accepted are 2389


In [23]:
hdl_clean = hdl[hdl["missing_p"] == False]
len(hdl_clean)

5873

In [24]:
hdl_clean.to_csv('/home/maria/git/SOROLLA/Results/hdl_genetic_correlation_pcorrected.csv', index=False)
hdl.to_csv('/home/maria/git/SOROLLA/Results/hdl_genetic_correlation_pcorrected_all_missing_too.csv', index=False)